# MDI3003 Lab 05 — Tweet Sentiment Analysis (Core Notebook)

Name: Dinesh | Registration No.: 23MID0319 | Batch: 2023 | Date: 25.08.2026

**Note:** Replace `data/Tweets.csv` with the real Kaggle Twitter US Airline Sentiment download before final submission; this notebook currently runs on a seeded synthetic dataset generated to match the same schema (offline environment). No code changes are needed since the column schema is identical.

In [ ]:
import os, re, json, time, platform, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

SEED = 42
np.random.seed(SEED)
OUT = '../outputs'
FIG = '../figures'

DATA_PATH = '../data/Tweets.csv'
TEXT_COL = 'text'
TARGET_COL = 'airline_sentiment'
ID_COL = 'tweet_id'
ENTITY_COL = 'airline'

raw = pd.read_csv(DATA_PATH)
df = raw[[ID_COL, TEXT_COL, TARGET_COL, ENTITY_COL]].copy()
df = df.dropna(subset=[TEXT_COL, TARGET_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str)

def normalize_tweet(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text)
    text = re.sub(r'@\w+', ' <USER> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df[TEXT_COL].map(normalize_tweet)
dup_id = df[ID_COL].duplicated().sum()
dup_text = df['clean_text'].duplicated().sum()

# --- EDA plots ---
plt.figure(figsize=(5,4))
df[TARGET_COL].value_counts().reindex(['negative','neutral','positive']).plot(kind='bar', color=['#c0392b','#7f8c8d','#27ae60'])
plt.title('Class Distribution'); plt.ylabel('Count'); plt.tight_layout()
plt.savefig(f'{FIG}/class_distribution.png', dpi=140); plt.close()

df['tw_len'] = df['clean_text'].str.split().apply(len)
plt.figure(figsize=(5,4))
plt.hist(df['tw_len'], bins=20, color='#2980b9')
plt.title('Tweet Length Distribution (tokens)'); plt.xlabel('Tokens'); plt.ylabel('Frequency'); plt.tight_layout()
plt.savefig(f'{FIG}/length_distribution.png', dpi=140); plt.close()

# --- split ---
train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df[TARGET_COL])
train_df.to_csv(f'{OUT}/train_manifest.csv', index=False)
test_df.to_csv(f'{OUT}/test_manifest.csv', index=False)
X_train, y_train = train_df['clean_text'], train_df[TARGET_COL]
X_test, y_test = test_df['clean_text'], test_df[TARGET_COL]

# --- Dummy baseline ---
dummy = Pipeline([('tfidf', TfidfVectorizer(min_df=2)), ('clf', DummyClassifier(strategy='stratified', random_state=SEED))])
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_macro = f1_score(y_test, dummy_pred, average='macro')
dummy_weighted = f1_score(y_test, dummy_pred, average='weighted')

# --- Simple lexicon baseline (VADER unavailable offline; small hand-built lexicon substitute) ---
POS_WORDS = set("amazing great love loved best thanks thank kudos smooth free friendly upgraded impressed comfy nailed helpful happy recommend".split())
NEG_WORDS = set("cancelled furious ridiculous lost worst rude terrible delayed unacceptable frustrated bumped hidden never annoyed angry squished".split())
def lexicon_label(text):
    toks = re.findall(r"[a-zA-Z']+", text.lower())
    score = sum(1 for t in toks if t in POS_WORDS) - sum(1 for t in toks if t in NEG_WORDS)
    if score > 0: return 'positive'
    if score < 0: return 'negative'
    return 'neutral'
lex_pred = X_test.map(lexicon_label)
lex_macro = f1_score(y_test, lex_pred, average='macro')
lex_weighted = f1_score(y_test, lex_pred, average='weighted')

# --- Classical CV ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models = {
    'MultinomialNB': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.98, sublinear_tf=True)), ('clf', MultinomialNB(alpha=0.5))]),
    'LogisticRegression': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.98, sublinear_tf=True)), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))]),
    'LinearSVC': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.98, sublinear_tf=True)), ('clf', LinearSVC(class_weight='balanced', random_state=SEED))]),
}

rows = [{'model':'Dummy','macro_f1_mean':dummy_macro,'macro_f1_sd':0.0,'weighted_f1_mean':dummy_weighted,'accuracy_mean':accuracy_score(y_test,dummy_pred),'fit_time_mean':0.0},
        {'model':'VADER(lexicon)','macro_f1_mean':lex_macro,'macro_f1_sd':0.0,'weighted_f1_mean':lex_weighted,'accuracy_mean':accuracy_score(y_test,lex_pred),'fit_time_mean':0.0}]

fitted = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
        scoring={'macro_f1':'f1_macro','weighted_f1':'f1_weighted','accuracy':'accuracy'},
        n_jobs=-1, return_train_score=False)
    rows.append({'model':name,
        'macro_f1_mean': scores['test_macro_f1'].mean(),
        'macro_f1_sd': scores['test_macro_f1'].std(),
        'weighted_f1_mean': scores['test_weighted_f1'].mean(),
        'accuracy_mean': scores['test_accuracy'].mean(),
        'fit_time_mean': scores['fit_time'].mean()})
    pipe.fit(X_train, y_train)
    fitted[name] = pipe

cv_results = pd.DataFrame(rows)
cv_results.to_csv(f'{OUT}/cv_results.csv', index=False)

# CV comparison plot (classical only)
plt.figure(figsize=(5,4))
sub = cv_results[cv_results.model.isin(['MultinomialNB','LogisticRegression','LinearSVC'])]
plt.bar(sub['model'], sub['macro_f1_mean'], yerr=sub['macro_f1_sd'], color='#8e44ad')
plt.title('CV Macro F1 by Model'); plt.ylabel('Macro F1'); plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(f'{FIG}/cv_comparison.png', dpi=140); plt.close()

# --- Model selection (best classical by CV macro F1, excluding dummy/lexicon) ---
classical_only = cv_results[cv_results.model.isin(models.keys())].copy()
pref_order = {'LogisticRegression':0,'LinearSVC':1,'MultinomialNB':2}  # tie-break: prefer interpretable linear model
classical_only['pref'] = classical_only['model'].map(pref_order)
classical_only = classical_only.sort_values(['macro_f1_mean','pref'], ascending=[False, True])
best_name = classical_only.iloc[0]['model']
best_model = fitted[best_name]
pred = best_model.predict(X_test)

report = classification_report(y_test, pred, digits=4, output_dict=True)
report_txt = classification_report(y_test, pred, digits=4)
macro_f1_test = f1_score(y_test, pred, average='macro')
weighted_f1_test = f1_score(y_test, pred, average='weighted')
acc_test = accuracy_score(y_test, pred)

cm = confusion_matrix(y_test, pred, labels=['negative','neutral','positive'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

def plot_cm(mat, title, fname, fmt):
    plt.figure(figsize=(4.5,4))
    plt.imshow(mat, cmap='Blues')
    plt.title(title)
    labels=['negative','neutral','positive']
    plt.xticks(range(3), labels, rotation=30); plt.yticks(range(3), labels)
    for i in range(3):
        for j in range(3):
            plt.text(j, i, fmt.format(mat[i,j]), ha='center', va='center',
                      color='white' if mat[i,j] > mat.max()/2 else 'black')
    plt.colorbar(); plt.tight_layout()
    plt.savefig(fname, dpi=140); plt.close()

plot_cm(cm, f'Confusion Matrix (Counts) - {best_name}', f'{FIG}/cm_counts.png', '{:d}')
plot_cm(cm_norm, f'Confusion Matrix (Row-Normalized) - {best_name}', f'{FIG}/cm_norm.png', '{:.2f}')

# Per-class precision/recall/F1 plot
classes = ['negative','neutral','positive']
prec = [report[c]['precision'] for c in classes]
rec = [report[c]['recall'] for c in classes]
f1s = [report[c]['f1-score'] for c in classes]
x = np.arange(3); w=0.25
plt.figure(figsize=(5,4))
plt.bar(x-w, prec, width=w, label='Precision')
plt.bar(x, rec, width=w, label='Recall')
plt.bar(x+w, f1s, width=w, label='F1')
plt.xticks(x, classes); plt.legend(); plt.title(f'Per-Class Metrics - {best_name}')
plt.tight_layout(); plt.savefig(f'{FIG}/per_class_metrics.png', dpi=140); plt.close()

# Top terms by class (training data only) using LR coefficients if best is LR/SVC-like, else NB
tfidf_best = best_model.named_steps['tfidf']
clf_best = best_model.named_steps['clf']
feat_names = np.array(tfidf_best.get_feature_names_out())
top_terms = {}
if hasattr(clf_best, 'coef_'):
    for i, c in enumerate(clf_best.classes_):
        idx = np.argsort(clf_best.coef_[i])[-10:][::-1]
        top_terms[c] = list(feat_names[idx])
cv_results_path = f'{OUT}/cv_results.csv'

# Test predictions & error analysis
pred_df = test_df[[ID_COL, TEXT_COL, TARGET_COL, ENTITY_COL]].copy()
pred_df['prediction'] = pred
pred_df.to_csv(f'{OUT}/test_predictions.csv', index=False)

errors = pred_df[pred_df[TARGET_COL] != pred_df['prediction']].copy()
errors_sample = errors.sample(min(10, len(errors)), random_state=SEED)
errors_sample.to_csv(f'{OUT}/error_analysis.csv', index=False)

# Entity-level analysis (min support 30)
entity_counts = pred_df[ENTITY_COL].value_counts()
valid_entities = entity_counts[entity_counts >= 30].index.tolist()
entity_dist = pd.crosstab(pred_df[ENTITY_COL], pred_df['prediction'], normalize='index').round(3)
entity_dist = entity_dist.loc[entity_dist.index.isin(valid_entities)]
entity_dist['N'] = entity_counts.loc[entity_dist.index]
entity_dist.to_csv(f'{OUT}/entity_sentiment_distribution.csv')

# Entity error rate
pred_df['correct'] = pred_df[TARGET_COL] == pred_df['prediction']
entity_err = pred_df.groupby(ENTITY_COL)['correct'].agg(['mean','count'])
entity_err = entity_err[entity_err['count'] >= 30]
entity_err['error_rate'] = 1 - entity_err['mean']
entity_err.to_csv(f'{OUT}/entity_error_rate.csv')

plt.figure(figsize=(6,4))
plt.bar(entity_err.index, entity_err['error_rate'], color='#e67e22')
plt.title('Error Rate by Airline (entity)'); plt.ylabel('Error rate'); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig(f'{FIG}/entity_error_rate.png', dpi=140); plt.close()

# Save selected pipeline & verify reload
os.makedirs('../models', exist_ok=True)
MODEL_PATH = '../models/selected_pipeline.joblib'
joblib.dump(best_model, MODEL_PATH)
reloaded = joblib.load(MODEL_PATH)
reload_ok = np.array_equal(reloaded.predict(X_test.iloc[:20]), best_model.predict(X_test.iloc[:20]))

summary = {
    'best_model': best_name,
    'dup_id': int(dup_id), 'dup_text': int(dup_text),
    'n_total': len(df), 'n_train': len(train_df), 'n_test': len(test_df),
    'dummy_macro_f1': dummy_macro, 'dummy_weighted_f1': dummy_weighted,
    'lex_macro_f1': lex_macro, 'lex_weighted_f1': lex_weighted,
    'macro_f1_test': macro_f1_test, 'weighted_f1_test': weighted_f1_test, 'acc_test': acc_test,
    'reload_ok': bool(reload_ok),
    'top_terms': top_terms,
    'class_report': report,
    'cm': cm.tolist(), 'cm_norm': cm_norm.tolist(),
}
with open(f'{OUT}/summary.json','w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2)[:3000])
print(report_txt)
print(cv_results)
print(entity_dist)
print(entity_err)
